In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          SELF-PRUNING NEURAL NETWORK — CIFAR-10 Classification               ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os
import json
import random
import warnings
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ══════════════════════════════════════════════════════════════════════════════
# 1.  REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════════

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ══════════════════════════════════════════════════════════════════════════════
# 2.  PRUNABLE LINEAR LAYER
# ══════════════════════════════════════════════════════════════════════════════

class PrunableLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, gate_init: float = 0.0) -> None:
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.weight      = nn.Parameter(torch.empty(out_features, in_features))
        self.bias        = nn.Parameter(torch.zeros(out_features))
        self.gate_scores = nn.Parameter(torch.full((out_features, in_features), gate_init))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates = torch.sigmoid(self.gate_scores)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self) -> torch.Tensor:
        return torch.sigmoid(self.gate_scores).detach().cpu()

    def layer_sparsity(self, threshold: float = 1e-2) -> float:
        g = self.get_gates()
        return (g < threshold).float().mean().item()

    def extra_repr(self) -> str:
        return f"in={self.in_features}, out={self.out_features}"


# ══════════════════════════════════════════════════════════════════════════════
# 3.  NETWORK ARCHITECTURE
# ══════════════════════════════════════════════════════════════════════════════

class SelfPruningNet(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        feat_dim = 256 * 4 * 4
        self.fc1 = PrunableLinear(feat_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = PrunableLinear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = PrunableLinear(256, num_classes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=0.4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

    def prunable_layers(self) -> List[PrunableLinear]:
        return [m for m in self.modules() if isinstance(m, PrunableLinear)]

    def sparsity_loss(self) -> torch.Tensor:
        return torch.cat([
            torch.sigmoid(layer.gate_scores).reshape(-1)
            for layer in self.prunable_layers()
        ]).sum()

    def overall_sparsity(self, threshold: float = 1e-2) -> float:
        total_pruned = total = 0
        for layer in self.prunable_layers():
            g = layer.get_gates()
            total_pruned += (g < threshold).sum().item()
            total        += g.numel()
        return total_pruned / total if total > 0 else 0.0

    def all_gates(self) -> torch.Tensor:
        return torch.cat([l.get_gates().reshape(-1) for l in self.prunable_layers()])

    def gate_param_ids(self) -> List[int]:
        return [id(l.gate_scores) for l in self.prunable_layers()]

    def count_parameters(self) -> Dict[str, int]:
        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        gate  = sum(l.gate_scores.numel() for l in self.prunable_layers())
        return {"total": total, "gate_params": gate, "weight_params": total - gate}


# ══════════════════════════════════════════════════════════════════════════════
# 4.  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def get_cifar10_loaders(data_dir="./data", batch_size=128, num_workers=2):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(), transforms.Normalize(mean, std),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(mean, std),
    ])
    train_set = torchvision.datasets.CIFAR10(root=data_dir, train=True,  download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_transform)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    print(f"  CIFAR-10: {len(train_set):,} train | {len(test_set):,} test | batch={batch_size}")
    return train_loader, test_loader


# ══════════════════════════════════════════════════════════════════════════════
# 5.  LAMBDA WARM-UP
# ══════════════════════════════════════════════════════════════════════════════

def get_effective_lambda(epoch, total_epochs, target_lam, warmup_frac=0.25, ramp_frac=0.25):
    warmup_end = int(total_epochs * warmup_frac)
    ramp_end   = int(total_epochs * (warmup_frac + ramp_frac))
    if epoch <= warmup_end:
        return 0.0
    elif epoch <= ramp_end:
        progress = (epoch - warmup_end) / max(ramp_end - warmup_end, 1)
        return target_lam * progress
    else:
        return target_lam


# ══════════════════════════════════════════════════════════════════════════════
# 6.  CHECKPOINT UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def checkpoint_path(output_dir: str, lam: float) -> str:
    """Path for per-lambda training checkpoint (latest state)."""
    return os.path.join(output_dir, f"ckpt_lam{lam:.1e}.pt")


def best_path(output_dir: str, lam: float) -> str:
    """Path for best-accuracy snapshot for a given lambda."""
    return os.path.join(output_dir, f"best_lam{lam:.1e}.pt")


def master_state_path(output_dir: str) -> str:
    """Path for global experiment progress (which lambdas are done)."""
    return os.path.join(output_dir, "experiment_state.json")


def load_master_state(output_dir: str) -> Dict:
    p = master_state_path(output_dir)
    if os.path.isfile(p):
        try:
            with open(p, "r") as f:
                return json.load(f)
        except Exception:
            return {"completed": {}, "results": []}
    return {"completed": {}, "results": []}


def save_master_state(output_dir: str, state: Dict) -> None:
    p = master_state_path(output_dir)
    tmp = p + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, p)


def save_checkpoint(path: str, payload: Dict) -> None:
    """Atomic save — write to tmp then rename, avoids corrupted files."""
    tmp = path + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, path)


# ══════════════════════════════════════════════════════════════════════════════
# 7.  TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, criterion, lam, device):
    model.train()
    sum_cls = sum_spar = sum_total = correct = total = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        cls_loss  = criterion(logits, labels)
        spar_loss = model.sparsity_loss()
        loss      = cls_loss + lam * spar_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        bs = images.size(0)
        sum_cls   += cls_loss.item()  * bs
        sum_spar  += spar_loss.item() * bs
        sum_total += loss.item()      * bs
        correct   += (logits.argmax(1) == labels).sum().item()
        total     += bs
    return {"cls_loss": sum_cls/total, "spar_loss": sum_spar/total,
            "total_loss": sum_total/total, "train_acc": correct/total}


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total


def train_model(lam, train_loader, test_loader, device, output_dir,
                epochs=60, lr=3e-4, gate_lr_mult=10.0, seed=42):
    """
    Training with full resume support.
    Saves checkpoint after EVERY epoch. If interrupted, resumes from last epoch.
    """
    set_seed(seed)

    model = SelfPruningNet().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    gate_ids = set(model.gate_param_ids())
    gate_params = [p for p in model.parameters() if id(p) in gate_ids and p.requires_grad]
    rest_params = [p for p in model.parameters() if id(p) not in gate_ids and p.requires_grad]

    optimizer = optim.AdamW([
        {"params": rest_params, "lr": lr,                "weight_decay": 1e-4},
        {"params": gate_params, "lr": lr * gate_lr_mult, "weight_decay": 0.0},
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=1, eta_min=1e-6)

    history = {k: [] for k in
        ["cls_loss", "spar_loss", "total_loss",
         "train_acc", "test_acc", "sparsity", "effective_lam"]}

    start_epoch = 1
    best_acc = 0.0
    best_state = None

    # ── RESUME LOGIC ─────────────────────────────────────────────────────
    ckpt_path = checkpoint_path(output_dir, lam)
    if os.path.isfile(ckpt_path):
        try:
            print(f"  [RESUME] Found checkpoint {ckpt_path}")
            ckpt = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            scheduler.load_state_dict(ckpt["scheduler_state"])
            history = ckpt["history"]
            start_epoch = ckpt["epoch"] + 1
            best_acc = ckpt.get("best_acc", 0.0)
            best_state = ckpt.get("best_state", None)
            # Restore RNG states for reproducibility
            if "rng_state" in ckpt:
                torch.set_rng_state(ckpt["rng_state"].cpu() if torch.is_tensor(ckpt["rng_state"]) else ckpt["rng_state"])
            if "cuda_rng_state" in ckpt and torch.cuda.is_available() and ckpt["cuda_rng_state"] is not None:
                try:
                    torch.cuda.set_rng_state_all([s.cpu() if torch.is_tensor(s) else s for s in ckpt["cuda_rng_state"]])
                except Exception:
                    pass
            if "numpy_rng_state" in ckpt:
                np.random.set_state(ckpt["numpy_rng_state"])
            if "python_rng_state" in ckpt:
                random.setstate(ckpt["python_rng_state"])
            print(f"  [RESUME] Continuing from epoch {start_epoch} | best_acc so far: {best_acc*100:.2f}%")
        except Exception as e:
            print(f"  [RESUME] Failed to load checkpoint ({e}). Starting fresh.")
            start_epoch = 1

    counts = model.count_parameters()
    print(f"\n{'='*62}")
    print(f"  lambda={lam:.1e} | total params: {counts['total']:,} | "
          f"gate params: {counts['gate_params']:,}")
    print(f"{'='*62}")
    print(f"  {'Ep':>4}  {'lam_eff':>8}  {'CLS':>7}  {'SPAR':>8}  "
          f"{'Train%':>7}  {'Test%':>7}  {'Sparse%':>8}")
    print(f"  {'-'*58}")

    if start_epoch > epochs:
        print(f"  [SKIP] Already completed all {epochs} epochs for this lambda.")
        if best_state is not None:
            model.load_state_dict(best_state)
        return model, history

    for epoch in range(start_epoch, epochs + 1):
        eff_lam = get_effective_lambda(epoch, epochs, lam)

        stats = train_one_epoch(model, train_loader, optimizer, criterion, eff_lam, device)
        test_acc = evaluate(model, test_loader, device)
        sparsity = model.overall_sparsity()
        scheduler.step()

        for k in ("cls_loss", "spar_loss", "total_loss", "train_acc"):
            history[k].append(stats[k])
        history["test_acc"].append(test_acc)
        history["sparsity"].append(sparsity)
        history["effective_lam"].append(eff_lam)

        if test_acc > best_acc:
            best_acc = test_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            # Save best snapshot immediately
            save_checkpoint(best_path(output_dir, lam), {
                "model_state": best_state, "accuracy": best_acc,
                "sparsity": sparsity, "lambda": lam, "epoch": epoch,
            })

        # ── SAVE CHECKPOINT EVERY EPOCH ──────────────────────────────────
        payload = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "history": history,
            "best_acc": best_acc,
            "best_state": best_state,
            "lambda": lam,
            "rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "numpy_rng_state": np.random.get_state(),
            "python_rng_state": random.getstate(),
        }
        save_checkpoint(ckpt_path, payload)

        if epoch % 10 == 0 or epoch in (1, 5) or epoch == epochs:
            print(f"  {epoch:>4}  {eff_lam:>8.1e}  "
                  f"{stats['cls_loss']:>7.4f}  {stats['spar_loss']:>8.0f}  "
                  f"{stats['train_acc']*100:>6.2f}%  "
                  f"{test_acc*100:>6.2f}%  "
                  f"{sparsity*100:>7.2f}%")

    if best_state is not None:
        model.load_state_dict(best_state)

    final_acc = evaluate(model, test_loader, device)
    final_spar = model.overall_sparsity()

    g_acc = ("EXCELLENT" if final_acc >= 0.53 else "GOOD" if final_acc >= 0.50
             else "MIN" if final_acc >= 0.48 else "BELOW")
    g_spar = ("EXCELLENT" if final_spar >= 0.85 else "GOOD" if final_spar >= 0.70
              else "MIN" if final_spar >= 0.50 else "BELOW")
    print(f"\n  Final: Acc={final_acc*100:.2f}% [{g_acc}] | "
          f"Sparsity={final_spar*100:.2f}% [{g_spar}]")

    return model, history


# ══════════════════════════════════════════════════════════════════════════════
# 8.  VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

BG = "#0d1117"; PANEL = "#161b22"; BORDER = "#30363d"
ACCENT = ["#58a6ff", "#3fb950", "#f78166", "#d2a8ff"]
TEXT = "#e6edf3"; SUB = "#8b949e"


def _style(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=SUB, labelsize=9)
    for s in ax.spines.values():
        s.set_color(BORDER)
    if title:  ax.set_title(title, color=TEXT, fontsize=11, fontweight="bold", pad=8)
    if xlabel: ax.set_xlabel(xlabel, color=SUB, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=SUB, fontsize=9)


def plot_gate_distribution(model, lam, path):
    layers = model.prunable_layers()
    labels = ["Layer 1 (4096->512)", "Layer 2 (512->256)", "Layer 3 (256->10)"]
    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    gs = fig.add_gridspec(2, 4, hspace=0.45, wspace=0.35,
                          left=0.06, right=0.97, top=0.88, bottom=0.1)
    for col, (layer, label) in enumerate(zip(layers, labels)):
        ax = fig.add_subplot(gs[0, col])
        g = layer.get_gates().numpy().flatten()
        sp = (g < 0.01).mean() * 100
        ax.hist(g, bins=80, color=ACCENT[col], edgecolor=BG, alpha=0.88, lw=0.3)
        ax.axvline(0.01, color=ACCENT[2], ls="--", lw=1.8, label="Threshold 0.01")
        _style(ax, title=f"{label}\nSparsity {sp:.1f}%", xlabel="Gate value", ylabel="Count")
        ax.text(0.97, 0.93, f"{sp:.1f}% pruned", transform=ax.transAxes,
                ha="right", va="top", color=ACCENT[2], fontsize=9, fontweight="bold")
        ax.legend(fontsize=7, facecolor=PANEL, labelcolor=SUB, framealpha=0.7, loc="upper center")
    ax_all = fig.add_subplot(gs[0, 3])
    all_g = model.all_gates().numpy()
    tot_sp = (all_g < 0.01).mean() * 100
    ax_all.hist(all_g, bins=100, color=ACCENT[3], edgecolor=BG, alpha=0.88, lw=0.3)
    ax_all.axvline(0.01, color=ACCENT[2], ls="--", lw=1.8)
    _style(ax_all, title=f"All Layers Combined\nSparsity {tot_sp:.1f}%",
           xlabel="Gate value", ylabel="Count")
    ax_all.text(0.97, 0.93, f"{tot_sp:.1f}% pruned", transform=ax_all.transAxes,
                ha="right", va="top", color=ACCENT[2], fontsize=9, fontweight="bold")
    ax_z = fig.add_subplot(gs[1, :2])
    ax_z.hist(all_g, bins=200, range=(0, 0.05), color=ACCENT[0], edgecolor=BG, alpha=0.88, lw=0.3)
    ax_z.axvline(0.01, color=ACCENT[2], ls="--", lw=1.8, label="Prune threshold")
    _style(ax_z, title="Zoomed: Near-Zero Gates [0, 0.05]", xlabel="Gate value", ylabel="Count")
    ax_z.legend(fontsize=8, facecolor=PANEL, labelcolor=SUB)
    ax_c = fig.add_subplot(gs[1, 2:])
    s_g = np.sort(all_g)
    cdf = np.arange(1, len(s_g) + 1) / len(s_g)
    ax_c.plot(s_g, cdf, color=ACCENT[1], lw=2)
    ax_c.axvline(0.01, color=ACCENT[2], ls="--", lw=1.8, label="Prune threshold")
    ax_c.axhline(tot_sp/100, color=ACCENT[3], ls=":", lw=1.5, label=f"Sparsity={tot_sp:.1f}%")
    _style(ax_c, title="Cumulative Distribution of Gate Values", xlabel="Gate value", ylabel="CDF")
    ax_c.legend(fontsize=8, facecolor=PANEL, labelcolor=SUB)
    fig.suptitle(f"Self-Pruning Network — Gate Distributions  (lambda = {lam:.1e})",
                 color=TEXT, fontsize=14, fontweight="bold")
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  Gate distribution -> {path}")


def plot_training_curves(all_histories, path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
    for ax in axes:
        ax.set_facecolor(PANEL)
        ax.tick_params(colors=SUB)
        for s in ax.spines.values():
            s.set_color(BORDER)
    for i, (lam, hist) in enumerate(all_histories.items()):
        c = ACCENT[i % len(ACCENT)]
        epochs = range(1, len(hist["test_acc"]) + 1)
        acc = [a*100 for a in hist["test_acc"]]
        spar = [s*100 for s in hist["sparsity"]]
        axes[0].plot(epochs, acc, color=c, lw=2, label=f"lam={lam:.1e} (final {acc[-1]:.1f}%)")
        axes[1].plot(epochs, spar, color=c, lw=2, label=f"lam={lam:.1e} (final {spar[-1]:.1f}%)")
    axes[0].axhline(53, color=ACCENT[1], ls=":", lw=1.2, alpha=0.7, label="Excellent (53%)")
    axes[0].axhline(50, color=ACCENT[2], ls=":", lw=1.0, alpha=0.5, label="Good (50%)")
    axes[1].axhline(85, color=ACCENT[1], ls=":", lw=1.2, alpha=0.7, label="Excellent (85%)")
    axes[1].axhline(70, color=ACCENT[2], ls=":", lw=1.0, alpha=0.5, label="Good (70%)")
    axes[0].set_title("Test Accuracy vs Epoch", color=TEXT, fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch", color=SUB); axes[0].set_ylabel("Accuracy (%)", color=SUB)
    axes[0].legend(facecolor=PANEL, labelcolor=TEXT, fontsize=8)
    axes[1].set_title("Sparsity vs Epoch", color=TEXT, fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch", color=SUB); axes[1].set_ylabel("Sparsity (%)", color=SUB)
    axes[1].legend(facecolor=PANEL, labelcolor=TEXT, fontsize=8)
    fig.suptitle("Self-Pruning Network — Training Dynamics",
                 color=TEXT, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  Training curves   -> {path}")


def plot_lambda_tradeoff(results, path):
    fig, ax = plt.subplots(figsize=(9, 7), facecolor=BG)
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=SUB)
    for s in ax.spines.values():
        s.set_color(BORDER)
    ax.axhspan(53, 100, alpha=0.07, color=ACCENT[1], label="Acc >= 53% zone")
    ax.axvspan(85, 100, alpha=0.07, color=ACCENT[0], label="Sparsity >= 85% zone")
    ax.axhline(53, color=ACCENT[1], ls="--", lw=1.5, alpha=0.6)
    ax.axhline(50, color=ACCENT[2], ls="--", lw=1.0, alpha=0.5)
    ax.axvline(85, color=ACCENT[0], ls="--", lw=1.5, alpha=0.6)
    ax.axvline(70, color=ACCENT[3], ls="--", lw=1.0, alpha=0.5)
    spars = [r["sparsity"]*100 for r in results]
    accs = [r["accuracy"]*100 for r in results]
    ax.plot(spars, accs, color=BORDER, lw=1.5, ls="-", zorder=2, alpha=0.6)
    for i, r in enumerate(results):
        sp = r["sparsity"]*100; ac = r["accuracy"]*100
        c = ACCENT[i % len(ACCENT)]
        ax.scatter(sp, ac, s=220, color=c, zorder=5, edgecolors=TEXT, linewidth=1.5)
        ax.annotate(f"lam={r['lambda']:.1e}\n({sp:.1f}%, {ac:.1f}%)",
                    (sp, ac), textcoords="offset points", xytext=(12, -8),
                    color=c, fontsize=9, fontweight="bold",
                    arrowprops=dict(arrowstyle="-", color=c, lw=0.8))
    ax.set_xlabel("Sparsity (%)", color=SUB, fontsize=12)
    ax.set_ylabel("Test Accuracy (%)", color=SUB, fontsize=12)
    ax.set_title("Accuracy vs Sparsity Trade-off  (lambda Ablation Study)\n"
                 "Upper-right corner = Excellent on BOTH metrics",
                 color=TEXT, fontsize=12, fontweight="bold")
    ax.legend(facecolor=PANEL, labelcolor=TEXT, fontsize=9, loc="lower left")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  Trade-off plot    -> {path}")


def print_results_table(results):
    print(f"\n{'='*68}")
    print(f"  {'RESULTS SUMMARY':^64}")
    print(f"{'='*68}")
    print(f"  {'Lambda':>10}  {'Test Acc':>12}  {'Sparsity':>12}  {'Acc Grade':>12}  {'Spar Grade':>12}")
    print(f"  {'-'*66}")
    for r in results:
        acc = r["accuracy"]; spar = r["sparsity"]
        ga = ("Excellent" if acc >= 0.53 else "Good" if acc >= 0.50
              else "Min" if acc >= 0.48 else "Below")
        gs = ("Excellent" if spar >= 0.85 else "Good" if spar >= 0.70
              else "Min" if spar >= 0.50 else "Below")
        print(f"  {r['lambda']:>10.1e}  {acc*100:>11.2f}%  "
              f"{spar*100:>11.2f}%  {ga:>12}  {gs:>12}")
    print(f"{'='*68}\n")


# ══════════════════════════════════════════════════════════════════════════════
# 9.  MAIN EXPERIMENT RUNNER WITH RESUME
# ══════════════════════════════════════════════════════════════════════════════

def run_experiments(
    lambdas:    List[float] = [1e-6, 1e-5, 5e-5],
    epochs:     int   = 60,
    batch_size: int   = 128,
    data_dir:   str   = "./data",
    output_dir: str   = "./outputs",
    seed:       int   = 42,
) -> None:
    os.makedirs(output_dir, exist_ok=True)
    set_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*62}")
    print(f"  Device : {device}")
    print(f"  Output : {output_dir}")
    print(f"  Targets: Accuracy >= 53% | Sparsity >= 85%")
    print(f"{'='*62}")

    # Load master state (which lambdas are already done)
    master = load_master_state(output_dir)
    completed = master.get("completed", {})  # {lam_str: {"accuracy":..., "sparsity":...}}
    print(f"  [STATE] {len(completed)} lambda(s) already completed: "
          f"{list(completed.keys()) if completed else 'none'}")

    train_loader, test_loader = get_cifar10_loaders(data_dir=data_dir, batch_size=batch_size)

    results: List[Dict] = []
    all_histories: Dict[float, Dict] = {}
    best_model = None
    best_score = -1.0
    best_lam = None

    for lam in lambdas:
        lam_key = f"{lam:.1e}"

        if lam_key in completed:
            print(f"\n[SKIP] lambda={lam_key} already fully completed. Loading result.")
            rec = completed[lam_key]
            results.append({"lambda": lam, "accuracy": rec["accuracy"], "sparsity": rec["sparsity"]})
            if "history" in rec:
                all_histories[lam] = rec["history"]
            # Load best model for plotting if needed
            bp = best_path(output_dir, lam)
            if os.path.isfile(bp):
                tmp_model = SelfPruningNet().to(device)
                state = torch.load(bp, map_location=device)
                tmp_model.load_state_dict(state["model_state"])
                score = rec["accuracy"] * 0.6 + rec["sparsity"] * 0.4
                if score > best_score:
                    best_score = score
                    best_model = tmp_model
                    best_lam = lam
            continue

        model, history = train_model(
            lam=lam, train_loader=train_loader, test_loader=test_loader,
            device=device, output_dir=output_dir, epochs=epochs, seed=seed)

        final_acc = history["test_acc"][-1] if history["test_acc"] else 0.0
        final_spar = history["sparsity"][-1] if history["sparsity"] else 0.0

        results.append({"lambda": lam, "accuracy": final_acc, "sparsity": final_spar})
        all_histories[lam] = history

        # Save final model snapshot
        torch.save({"model_state": model.state_dict(), "accuracy": final_acc,
                    "sparsity": final_spar, "lambda": lam},
                   os.path.join(output_dir, f"model_lam{lam:.1e}.pt"))

        # Mark this lambda as completed and persist
        completed[lam_key] = {
            "accuracy": final_acc, "sparsity": final_spar,
            "lambda": lam, "history": history,
        }
        master["completed"] = completed
        master["results"] = results
        save_master_state(output_dir, master)

        # Optional: remove the intermediate training checkpoint to save disk space
        ckpt_file = checkpoint_path(output_dir, lam)
        if os.path.isfile(ckpt_file):
            try:
                os.remove(ckpt_file)
                print(f"  [CLEANUP] Removed training checkpoint {ckpt_file}")
            except Exception:
                pass

        score = final_acc * 0.6 + final_spar * 0.4
        if score > best_score:
            best_score = score
            best_model = model
            best_lam = lam

    print_results_table(results)

    if best_model is not None and best_lam is not None:
        plot_gate_distribution(best_model, best_lam,
                               os.path.join(output_dir, "gate_distribution.png"))
    if all_histories:
        plot_training_curves(all_histories,
                             os.path.join(output_dir, "training_curves.png"))
    if results:
        plot_lambda_tradeoff(results,
                             os.path.join(output_dir, "lambda_tradeoff.png"))

    print(f"\nAll experiments complete. Outputs -> {output_dir}/")


# ══════════════════════════════════════════════════════════════════════════════
# 10.  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    set_seed(42)
    # TIP for Google Colab: point output_dir to Google Drive so checkpoints
    # survive runtime disconnects. Example:
    #   from google.colab import drive
    #   drive.mount('/content/drive')
    #   output_dir = "/content/drive/MyDrive/selfprune_outputs"
    run_experiments(
        lambdas    = [1e-6, 1e-5, 5e-5],
        epochs     = 60,
        batch_size = 128,
        data_dir   = "./data",
        output_dir = "/content/drive/MyDrive/selfprune_outputs",
    )


  Device : cuda
  Output : /content/drive/MyDrive/selfprune_outputs
  Targets: Accuracy >= 53% | Sparsity >= 85%
  [STATE] 0 lambda(s) already completed: none


100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s]


  CIFAR-10: 50,000 train | 10,000 test | batch=128

  lambda=1.0e-06 | total params: 5,019,850 | gate params: 2,230,784
    Ep   lam_eff      CLS      SPAR   Train%    Test%   Sparse%
  ----------------------------------------------------------
     1   0.0e+00   1.6047   1114506   46.07%   66.02%     0.00%
     5   0.0e+00   0.9201   1109311   75.32%   80.41%     0.00%
    10   0.0e+00   0.7554   1104839   82.27%   84.95%     0.00%
    20   3.3e-07   0.6217   1098078   87.70%   88.10%     0.00%
    30   1.0e-06   0.5952    691089   88.51%   88.73%     0.00%
    40   1.0e-06   0.5129    588915   92.16%   90.08%     0.28%
    50   1.0e-06   0.5171    396491   91.80%   89.98%    10.03%
    60   1.0e-06   0.4620    360120   94.15%   90.88%    13.66%

  Final: Acc=90.97% [EXCELLENT] | Sparsity=13.64% [BELOW]
  [CLEANUP] Removed training checkpoint /content/drive/MyDrive/selfprune_outputs/ckpt_lam1.0e-06.pt

  lambda=1.0e-05 | total params: 5,019,850 | gate params: 2,230,784
    Ep   lam_ef